# VFSS mask range debug

Goal: understand why segmentation masks in `diffusion_row` can show a grey background during training.

Hypothesis: the active VFSS dataloader may be giving the diffusion first-stage model masks in `0/1` space instead of the image-style `-1/1` space expected by SDSeg training and image logging.

## Reasoning map

1. `configs/SDSeg/vfss-new-inca.yaml` sets `first_stage_key: "segmentation"`, so the mask is the diffusion target.
2. `SDSeg.get_input(...)` routes that tensor through the first-stage autoencoder before diffusion training.
3. `SDSeg.log_images(...)` logs `inputs`, `reconstruction`, and `diffusion_row` from this same mask latent path.
4. `main.ImageLogger.log_local(...)` saves logged tensors as `(grid + 1) / 2`. Therefore a raw mask background value of `0` displays as `0.5`, i.e. grey. A normalized background value of `-1` displays as `0`, i.e. black.

So the important check is not only `min >= -1 and max <= 1`. A binary `0/1` mask passes that loose check, but it is still centered incorrectly for the first-stage image model and for the logger convention.

In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import torch
import yaml

# Make this notebook work when launched either from the repo root or from notebooks/.
repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

CONFIG_PATH = repo / "configs" / "SDSeg" / "vfss-new-inca.yaml"
print(repo)
print(CONFIG_PATH)

/data_ssd/caioseda/projetos/Stable-Diffusion-Seg
/data_ssd/caioseda/projetos/Stable-Diffusion-Seg/configs/SDSeg/vfss-new-inca.yaml


/home/caioseda/miniconda3/envs/sdseg-cpython/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with CONFIG_PATH.open() as f:
    cfg = yaml.safe_load(f)

model_params = cfg["model"]["params"]
data_params = cfg["data"]["params"]

print("model target:", cfg["model"]["target"])
print("first_stage_key:", model_params["first_stage_key"])
print("cond_stage_key:", model_params["cond_stage_key"])
print("train target:", data_params["train"]["target"])
print("validation target:", data_params["validation"]["target"])
print("test target:", data_params["test"]["target"])

model target: ldm.models.diffusion.SDSeg.SDSeg
first_stage_key: segmentation
cond_stage_key: image
train target: ldm.data.vfss_new.VFSSIncaTrain
validation target: ldm.data.vfss_new.VFSSIncaVal
test target: ldm.data.vfss_new.VFSSIncaTest


The active config uses `ldm.data.vfss_new.VFSSIncaTrain`, not the older `ldm.data.vfss.VFSSTrain`. That matters because the older loader explicitly maps binary masks to `-1/1`, while `vfss_new` currently thresholds to `0/1`.

In [3]:
from ldm.util import instantiate_from_config

datasets = {
    "train": instantiate_from_config(data_params["train"]),
    "validation": instantiate_from_config(data_params["validation"]),
    "test": instantiate_from_config(data_params["test"]),
}

for split, ds in datasets.items():
    print(split, type(ds), len(ds))

[Dataset]: Initializing VFSSIncaTrain with parameters: {'dataset_path': '/data_ssd/caioseda/data/dataset_inca_all_frames/', 'size': 256, 'window_size': 1, 'stride': 1, 'repeat_channels': True}
train <class 'ldm.data.vfss_new.VFSSIncaTrain'> 568
validation <class 'ldm.data.vfss_new.VFSSIncaVal'> 80
test <class 'ldm.data.vfss_new.VFSSIncaTest'> 161


In [4]:
def as_tensor(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu()
    return torch.as_tensor(x)


def sample_stats(sample):
    rows = []
    for key in ["image", "segmentation"]:
        x = as_tensor(sample[key])
        rows.append(
            {
                "key": key,
                "shape": tuple(x.shape),
                "dtype": str(x.dtype),
                "min": float(x.min()),
                "max": float(x.max()),
                "unique_first_20": torch.unique(x).tolist()[:20],
            }
        )
    return pd.DataFrame(rows)


sample = datasets["train"][0]
sample_stats(sample)

,key,shape,dtype,min,max,unique_first_20
0,image,"(256, 256, 3)",torch.float32,-1.0,1.0,"[-1.0, -0.9450980424880981, -0.943137288093566..."
1,segmentation,"(256, 256, 3)",torch.float32,-1.0,1.0,"[-1.0, 1.0]"


Expected result from the current repo state: `image` is `float32` in `[-1, 1]`, but `segmentation` is `int64` with unique values `[0, 1]`.

In [5]:
def scan_mask_range(ds, key="segmentation", limit=None):
    n = len(ds) if limit is None else min(limit, len(ds))
    mins, maxs = [], []
    dtypes, shapes, unique_values = set(), set(), set()
    outside_loose_range = []
    not_pm_one = []
    empty_masks = []

    for i in range(n):
        x = as_tensor(ds[i][key])
        dtypes.add(str(x.dtype))
        shapes.add(tuple(x.shape))
        mn = float(x.min())
        mx = float(x.max())
        mins.append(mn)
        maxs.append(mx)

        if mn < -1.0 or mx > 1.0:
            outside_loose_range.append((i, mn, mx))

        unique = torch.unique(x).tolist()
        unique_values.update(int(v) if float(v).is_integer() else float(v) for v in unique)

        # Strict endpoint check for binary masks used as image-like first-stage input.
        if not set(unique).issubset({-1, 1}):
            not_pm_one.append((i, unique[:10]))

        if mx == 0:
            empty_masks.append(i)

    return {
        "n_scanned": n,
        "dtype": sorted(dtypes),
        "shape": sorted(shapes),
        "global_min": min(mins),
        "global_max": max(maxs),
        "unique_values": sorted(unique_values),
        "outside_-1_1_count": len(outside_loose_range),
        "outside_-1_1_first": outside_loose_range[:5],
        "not_binary_minus1_plus1_count": len(not_pm_one),
        "not_binary_minus1_plus1_first": not_pm_one[:5],
        "empty_mask_count": len(empty_masks),
        "empty_mask_first": empty_masks[:10],
    }


scan_results = {split: scan_mask_range(ds) for split, ds in datasets.items()}
pd.DataFrame(scan_results).T

,n_scanned,dtype,shape,global_min,global_max,unique_values,outside_-1_1_count,outside_-1_1_first,not_binary_minus1_plus1_count,not_binary_minus1_plus1_first,empty_mask_count,empty_mask_first
train,568,[torch.float32],"[(256, 256, 3)]",-1.0,1.0,"[-1, 1]",0,[],0,[],0,[]
validation,80,[torch.float32],"[(256, 256, 3)]",-1.0,1.0,"[-1, 1]",0,[],0,[],0,[]
test,161,[torch.float32],"[(256, 256, 3)]",-1.0,1.0,"[-1, 1]",0,[],0,[],0,[]


Observed from a train split run in this workspace:

- `n = 568`
- `dtype = ['torch.int64']`
- `shape = [(256, 256, 3)]`
- `global_min = 0.0`, `global_max = 1.0`
- `unique_values = [0, 1]`
- `outside_-1_1_count = 0`
- `not_binary_minus1_plus1_count = 568`

Interpretation: the masks are constrained inside `[-1, 1]` in the loose sense, but they are not normalized to `-1/1`. Every scanned train mask uses `0` for background.

In [ ]:
import matplotlib.pyplot as plt

seg = as_tensor(datasets["train"][0]["segmentation"]).float()
raw_ch0 = seg[..., 0]

# This is the display conversion used by main.ImageLogger.log_local.
logger_display_from_current = ((seg + 1.0) / 2.0)[..., 0]

# This is what the same mask would become if converted from 0/1 to -1/1 before logging.
seg_pm_one = (seg * 2.0) - 1.0
logger_display_after_fix = ((seg_pm_one + 1.0) / 2.0)[..., 0]

print("raw unique:", torch.unique(seg).tolist())
print("logger display unique from current 0/1 mask:", torch.unique(logger_display_from_current).tolist())
print("logger display unique after 0/1 -> -1/1 conversion:", torch.unique(logger_display_after_fix).tolist())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(raw_ch0, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("raw mask channel\n0/1")
axes[1].imshow(logger_display_from_current, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("ImageLogger view\ncurrent mask")
axes[2].imshow(logger_display_after_fix, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("ImageLogger view\nafter -1/1 fix")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

This explains the grey background without needing the model to be wrong yet: if a mask background is `0`, the logger maps it to `(0 + 1) / 2 = 0.5`, which is grey. If the background is `-1`, the logger maps it to `0`, which is black.

The same mismatch also affects the first-stage autoencoder input. The first-stage model is configured with `in_channels: 3` and is used like an image autoencoder. Existing dataset classes generally feed training masks in `[-1, 1]`, not `0/1`, before encoding them into latent diffusion targets.

In [ ]:
import inspect
from ldm.data.vfss_new import VFSSWindowImageDataset

print(inspect.getsource(VFSSWindowImageDataset._VFSSWindowImageDataset__preprocess_mask))

The key line is:

```python
mask = (mask > 0).long()
```

Later, `__getitem__` repeats that mask to three channels:

```python
returns['segmentation'] = returns['segmentation'].unsqueeze(-1).repeat(1,1,3)
```

That produces an HWC `0/1` mask, exactly matching the scan.

## Downstream caution

Do not blindly change every VFSS split to `-1/1` without checking evaluation. `SDSeg.log_dice(...)` treats `prompts["segmentation"]` as class labels and asserts `label.max() == self.num_classes - 1`; for binary segmentation that means it expects `0/1` labels when computing Dice.

This repo has precedent for split-dependent semantics: for example, some older dataset classes return `-1/1` masks for train/val diffusion targets and raw `0/1` masks for test/evaluation labels. The current `vfss_new` loader does not make that distinction.

## Conclusion

The active dataloader is a plausible source of the grey-background `diffusion_row` problem.

- The masks are binary and inside `[-1, 1]`, so a loose range assertion passes.
- They are not normalized to `{-1, 1}`; background is `0`, not `-1`.
- The logger maps `0` to grey via `(x + 1) / 2`.
- The first-stage autoencoder also receives these masks as image-like inputs, so the diffusion target latent is built from a mid-grey background instead of a black background.

Recommended fix path: make the training/validation diffusion target mask explicitly `float32` in `-1/1` space, while preserving raw `0/1` labels for Dice evaluation. The cleanest design is either a split-aware conversion in `vfss_new` or separate keys such as `segmentation` for the normalized training target and `segmentation_label` for evaluation labels.